# Week 5 — Evaluation and Optimization (Notebook-only)

Goal: measure the current RAG from Week 2 to Week 4 without refactoring the architecture.
We do not change core logic; we only measure and compare.


## 1) Baseline
- Embedding: `hashing-768-stable`
- Index: `FAISS IndexFlatIP`
- Chunking: `chunk_size=300`, `chunk_overlap=50`
- Retrieval: `top_k=3`
- Prompt: strict grounded QA

Why we fix the baseline: otherwise we cannot tell what actually improved.


In [ ]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
import faiss
from sklearn.feature_extraction.text import HashingVectorizer
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

DATA_DIR = Path('../data') if Path('../data').exists() else Path('data')
CHUNK_SIZE = 300
CHUNK_OVERLAP = 50
TOP_K = 3
MAX_PAGES_PER_PDF = 20  # safety for notebook stability

def ingest_docs(data_dir: Path):
    docs = []
    errors = []
    for topic_dir in sorted(data_dir.iterdir()):
        if not topic_dir.is_dir():
            continue
        topic = topic_dir.name
        for pdf_path in sorted(topic_dir.glob('*.pdf')):
            try:
                pages = PyPDFLoader(str(pdf_path)).load()[:MAX_PAGES_PER_PDF]
                for p in pages:
                    docs.append({
                        'text': p.page_content,
                        'source': pdf_path.name,
                        'topic': topic,
                        'page': int(p.metadata.get('page', 0)),
                    })
            except Exception as e:
                errors.append({'source': pdf_path.name, 'error': str(e)})
    return docs, errors

def chunk_docs(docs, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=['\n\n', '\n', '. ', ' ', ''],
    )
    chunks = []
    for d in docs:
        parts = splitter.split_text(d['text'])
        for part in parts:
            chunks.append({
                'text': part,
                'source': d['source'],
                'topic': d['topic'],
                'page': d['page'],
            })
    return chunks

class HashEmbedder:
    def __init__(self, n_features=768):
        self.n_features = n_features
        self.v = HashingVectorizer(n_features=n_features, alternate_sign=False, norm='l2')

    def encode(self, texts):
        x = self.v.transform(texts).astype(np.float32).toarray()
        faiss.normalize_L2(x)
        return x

docs, ingest_errors = ingest_docs(DATA_DIR)
chunks = chunk_docs(docs)
print(f'Docs: {len(docs)}, Chunks: {len(chunks)}, Ingest errors: {len(ingest_errors)}')


In [ ]:
embedder = HashEmbedder(768)
texts = [c['text'] for c in chunks]
vecs = embedder.encode(texts)
index = faiss.IndexFlatIP(vecs.shape[1])
index.add(vecs)
print('Index size:', index.ntotal, 'dim:', vecs.shape[1])


## 2) Retrieval Latency

In [ ]:
QUERIES = [
    'What is RAG?',
    'How to create a git branch?',
    'What is Google Cloud Platform?',
]

rows = []
for q in QUERIES:
    for i in range(5):
        t0 = time.perf_counter()
        qv = embedder.encode([q])
        t1 = time.perf_counter()
        scores, idx = index.search(qv, TOP_K)
        t2 = time.perf_counter()
        rows.append({
            'query': q,
            'iter': i+1,
            'embed_ms': (t1-t0)*1000,
            'search_ms': (t2-t1)*1000,
            'total_ms': (t2-t0)*1000,
        })

lat_df = pd.DataFrame(rows)
summary = lat_df.groupby('query')[['embed_ms','search_ms','total_ms']].agg(['min','mean','max']).round(3)
display(summary)
out_art = Path('../artifacts') if Path('../artifacts').exists() else Path('artifacts')
out_art.mkdir(parents=True, exist_ok=True)
lat_df.to_csv(out_art / 'week5_latency_runs.csv', index=False)


## 3) Retrieval Relevance (manual labeling)
- In this dataset, relevance means the RAG/GIT/GCP topic matches the question.
- The table below is for manual relevance labels `relevant` (1/0).


In [ ]:
relevance_rows = []
for q in QUERIES:
    qv = embedder.encode([q])
    scores, idx = index.search(qv, TOP_K)
    for rank, (s, i) in enumerate(zip(scores[0], idx[0]), 1):
        c = chunks[int(i)]
        relevance_rows.append({
            'query': q,
            'rank': rank,
            'score': float(s),
            'topic': c['topic'],
            'source': c['source'],
            'text_preview': c['text'][:120].replace('\n',' '),
            'relevant': None,
        })

rel_df = pd.DataFrame(relevance_rows)
display(rel_df)
print('After manual labeling, set 1/0 in the relevant column and compute Hit@k')


In [ ]:
tmp = rel_df.copy()
tmp['relevant'] = pd.to_numeric(tmp['relevant'], errors='coerce')
if tmp['relevant'].notna().any():
    hit_k = tmp.groupby('query')['relevant'].max().mean()
    rel_ratio = tmp['relevant'].mean()
    print({'Hit@k': round(float(hit_k), 4), 'relevance_ratio': round(float(rel_ratio), 4)})
else:
    print('Manual labels are empty: fill rel_df["relevant"] first.')


## 4) Embedding Comparison (2 models)

In [ ]:
embed_cfgs = [('hash-384', 384), ('hash-768', 768)]
cmp_rows = []

for name, dim in embed_cfgs:
    emb = HashEmbedder(dim)
    t0 = time.perf_counter()
    vv = emb.encode(texts)
    t1 = time.perf_counter()
    ix = faiss.IndexFlatIP(vv.shape[1])
    ix.add(vv)
    for q in QUERIES:
        q0 = time.perf_counter()
        qv = emb.encode([q])
        q1 = time.perf_counter()
        scores, idx = ix.search(qv, TOP_K)
        q2 = time.perf_counter()
        top_topic = chunks[int(idx[0][0])]['topic'] if len(idx[0]) else None
        cmp_rows.append({
            'model': name,
            'dim': dim,
            'query': q,
            'build_ms': (t1-t0)*1000,
            'embed_ms': (q1-q0)*1000,
            'search_ms': (q2-q1)*1000,
            'top1_topic': top_topic,
            'top1_score': float(scores[0][0]) if len(scores[0]) else None,
        })

cmp_df = pd.DataFrame(cmp_rows)
display(cmp_df.groupby(['model','dim'])[['build_ms','embed_ms','search_ms','top1_score']].mean().round(3))


## 5) LLM Output Quality + Prompt Optimization

In [ ]:
from langchain_ollama import OllamaLLM

BASE_PROMPT = '''You are a RAG assistant.
Use ONLY context. If not enough info, say: I don't know based on provided context.

CONTEXT:
{context}

QUESTION: {query}
ANSWER:
'''

OPT_PROMPT = '''You are a strict grounded assistant.
Rules:
1) Use only CONTEXT chunks.
2) If evidence is missing/weak, answer exactly: Insufficient evidence in retrieved context.
3) Add citations like [Chunk 1], [Chunk 2].
4) Do not add external facts.

CONTEXT:
{context}

QUESTION: {query}
ANSWER:
'''

def retrieve_context(query, top_k=3):
    qv = embedder.encode([query])
    scores, idx = index.search(qv, top_k)
    ctx = []
    for r, i in enumerate(idx[0], 1):
        c = chunks[int(i)]
        ctx.append(f'[Chunk {r}] ' + c['text'][:500])
    return '

'.join(ctx), scores[0].tolist()

def try_llm(prompt):
    try:
        llm = OllamaLLM(model='gemma3:4b', base_url='http://localhost:11434', temperature=0.0, validate_model_on_init=True)
        return (llm.invoke(prompt) or '').strip(), None
    except Exception as e:
        return '', str(e)

q = 'What is RAG?'
context, s = retrieve_context(q, TOP_K)
p1 = BASE_PROMPT.format(context=context, query=q)
p2 = OPT_PROMPT.format(context=context, query=q)
a1, e1 = try_llm(p1)
a2, e2 = try_llm(p2)

out = pd.DataFrame([
    {'variant':'baseline','error':e1,'answer':a1[:500]},
    {'variant':'optimized','error':e2,'answer':a2[:500]},
])
display(out)
print('Context scores:', [round(x,4) for x in s])


## Final Intern Conclusions
1. What impacts latency the most: ...
2. What impacts retrieval quality the most: ...
3. Trade-off by embedding choice: ...
4. How prompt wording affects hallucination risk: ...
5. What to improve in Week 6: ...
